# Original course experiments
Historical group-project notebook. Outputs and execution counts have been cleared; the CSV path now points to `../data/abcnews-date-text.csv`.
See [methodology notes](../docs/methodology.md) for known experimental limitations. Use `scripts/cluster_headlines.py` for the maintained pipeline.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, silhouette_samples
from kneed import KneeLocator
import matplotlib.pyplot as plt
from collections import Counter
import re
from collections import defaultdict
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer
from nltk.tokenize import RegexpTokenizer
from collections import Counter
import seaborn as sns
import warnings
import umap
import hdbscan

warnings.filterwarnings('ignore')


sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

tokenizer = RegexpTokenizer(r'[a-zA-Z\']+')


Tokenization function

In [ ]:
def tokenize(text):
    return tokenizer.tokenize(text.lower())

In [ ]:
file_path = '../data/abcnews-date-text.csv'
n_samples = 50000


# load data
df = pd.read_csv(file_path)


# Remove Duplicate Titles
initial_count = len(df)
df = df.drop_duplicates(subset=['headline_text'], keep='first')
removed_count = initial_count - len(df)
print(f"Removed {removed_count:,} duplicate titles")
print(f"Remaining {len(df):,} records")

In [ ]:
# Extract year for stratified sampling
df['publish_date_str'] = df['publish_date'].astype(str)
df['year'] = df['publish_date_str'].str[:4]

# Calculate the number of samples to be drawn from each year (by proportion)
year_counts = df['year'].value_counts()
print(f"\nOriginal data year distribution:")
for year in sorted(year_counts.index):
    print(f"{year}: {year_counts[year]:,} records")

In [ ]:
# random sampling by year
if len(df) > n_samples:
    # Calculate sampling fraction
    sample_fraction = n_samples / len(df)
    print(f"\nSampling fraction: {sample_fraction * 100:.2f}%")
    
    # Stratified sampling
    df_sampled = df.groupby('year', group_keys=False).apply(
        lambda x: x.sample(frac=sample_fraction, random_state=42)
    ).reset_index(drop=True)

    
    # If the sample size is insufficient after sampling (due to rounding), supplement randomly
    if len(df_sampled) < n_samples:
        remaining = n_samples - len(df_sampled)
        df_remaining = df[~df.index.isin(df_sampled.index)]
        df_additional = df_remaining.sample(n=remaining, random_state=42)
        df_sampled = pd.concat([df_sampled, df_additional], ignore_index=True)
    
    # If the sample size exceeds, randomly reduce it
    elif len(df_sampled) > n_samples:
        df_sampled = df_sampled.sample(n=n_samples, random_state=42)
    
    df = df_sampled
    
    print(f"final data size: {len(df):,} records")

    
    # Display the year distribution after sampling
    sampled_year_counts = df['year'].value_counts()
    print(f"\nYear distribution after sampling:")
    for year in sorted(sampled_year_counts.index):
        original = year_counts[year]
        sampled = sampled_year_counts[year]
        percentage = (sampled / original * 100)
        print(f"{year}: {sampled:5d} records (constitutes {percentage:5.2f}% of that year)")
else:
    print(f"Insufficient data (less than {n_samples:,}), using all data")

In [ ]:
# Sort by date and extract data
df = df.sort_values('publish_date_str').reset_index(drop=True)

# Extract dates and headlines
dates = df['publish_date_str'].values
headlines = df['headline_text'].values

# Tokenization
tokenized_texts = [tokenize(text) for text in headlines]

print(f"Number of dates: {len(dates)}")
print(f"Number of headlines: {len(headlines)}")
print(f"Number of tokenized samples: {len(tokenized_texts)}")


In [ ]:
# Basic statistics
print(f"\n[1] Basic information about the data")
print(f"  Total data size: {len(headlines):,} records")
years = [d[:4] for d in dates]
print(f"Date range: {min(years)} - {max(years)}")

# Headline length statistics
print(f"\n[2] Headline length statistics")
lengths = [len(h.split()) for h in headlines]
print(f"  Average length: {np.mean(lengths):.2f} words")
print(f"  Median: {np.median(lengths):.0f} words")
print(f"  Range: {min(lengths)} - {max(lengths)} words")


In [ ]:
# Top 20 high-frequency words (without stop words)
print(f"\n[3] Top 20 high-frequency words (without stop words)")

# English stop words & custom noise words
useful_stopwords = set(text.ENGLISH_STOP_WORDS)
useful_stopwords.update([
    "said", "say", "says",
    "year", "new", "one",
    "two", "three",
    "also", "first",
    "could", "would", "nsw",
    "qld", "vic", "wa", "sa",
    "act", "nt", "tas",
    # Add common words that should not be topic keywords
    "the", "with", "and", "over", "after", "from", "into", "about", "more", "back", "out",
    # Add words that frequently appear but should not be topic keywords
    "new", "says", "not", "down", "off", "shows"
])


filtered_words = []
for tokens in tokenized_texts:
    for w in tokens:
        if w not in useful_stopwords and len(w) > 2:
            filtered_words.append(w)

word_freq = Counter(filtered_words)

for i, (word, count) in enumerate(word_freq.most_common(20), 1):
    print(f"  {i:2d}. {word:12s}: {count:6d}")

In [ ]:
# Figure 1: Headline length distribution
plt.figure(figsize=(7, 5))
plt.hist(lengths, bins=25, color='steelblue', alpha=0.7, edgecolor='black')
plt.axvline(np.mean(lengths), color='red', linestyle='--', linewidth=2,
            label=f'Mean: {np.mean(lengths):.1f}')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.title('Headline Length Distribution')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 2: Yearly distribution
plt.figure(figsize=(7, 5))
year_counts = pd.Series(years).value_counts().sort_index()
year_counts.plot(kind='bar', color='coral', alpha=0.8)
plt.xlabel('Year')
plt.ylabel('Number of Headlines')
plt.title('Headlines per Year')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Figure 3: Top 20 high-frequency words
plt.figure(figsize=(7, 5))
top15 = word_freq.most_common(20)
words, counts = zip(*top15)
y_pos = np.arange(len(words))

plt.barh(y_pos, counts, color='lightgreen', alpha=0.7)
plt.yticks(y_pos, words)
plt.gca().invert_yaxis()
plt.xlabel('Frequency')
plt.title('Top 20 Words (Filtered)')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print(f"Number of samples: {len(headlines):,}")
print(f"Time span: {min(years)} to {max(years)}")
print(f"Average length: {np.mean(lengths):.2f} words")
print(f"Vocabulary size (after filtering): {len(word_freq):,} words")

Feature engineering -- method1: tf-idf

In [ ]:
# Define stopwords (including punctuation)
punc = ['.', ',', '"', "'", '?', '!', ':', ';', '(', ')', '[', ']', '{', '}', "%"]
# Add common words that should not be topic keywords
additional_stopwords = {"the", "with", "and", "over", "after", "from", "into", "about", "more", "back", "out",
                        "new", "says", "not", "down", "off", "shows"}
stop_words = list(text.ENGLISH_STOP_WORDS.union(punc).union(additional_stopwords))

# TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer(
    stop_words=stop_words,
    tokenizer=tokenize,
    max_features=5000
)
tfidf_matrix = tfidf_vectorizer.fit_transform(headlines)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
word_features = tfidf_vectorizer.get_feature_names_out()
print(f"Number of feature words: {len(word_features)}")

In [ ]:
# LSA dimensionality reduction
n_components = 100
lsa = TruncatedSVD(n_components=n_components, random_state=42)
tfidf_lsa_features = lsa.fit_transform(tfidf_matrix)

print(f"Shape of LSA feature matrix: {tfidf_lsa_features.shape}")
print(f"Explained variance ratio: {lsa.explained_variance_ratio_.sum():.4f}")

method2: SBERT (Sentence-BERT)

In [ ]:
# Load SBERT model
# Using 'all-MiniLM-L6-v2' which is a fast and efficient model
# You can also use 'all-mpnet-base-v2' for better quality but slower
print("Loading SBERT model...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"SBERT model loaded. Embedding dimension: {sbert_model.get_sentence_embedding_dimension()}")


In [ ]:
# Generate SBERT embeddings for each headline
# SBERT directly encodes the entire sentence, no need for word-level averaging
print("Generating SBERT embeddings...")
sbert_features = sbert_model.encode(headlines, show_progress_bar=True, batch_size=32)

print(f"SBERT feature matrix completed.")
print(f"Shape of SBERT feature matrix: {sbert_features.shape}")
print(f"Embedding dimension: {sbert_features.shape[1]}")

Method3:  TF-IDF weighted SBERT  
combines the statistical significance of words (TF-IDF) with semantic representations (SBERT), allowing important words to hold greater weight in the document vector

In [ ]:

# TF-IDF weighted SBERT embeddings
# Since SBERT encodes entire sentences, we'll use a different approach:
# We'll weight the SBERT embeddings by the TF-IDF scores of important words in the sentence

tfidf_matrix = tfidf_vectorizer.transform(headlines)
feature_names = tfidf_vectorizer.get_feature_names_out()

# Use the SBERT embeddings already computed in Cell 18 (sbert_features)
# No need to recompute

# Calculate TF-IDF weighted SBERT embeddings
# Approach: Multiply SBERT embedding by a scalar weight based on TF-IDF scores
mixed_embeddings = []

for idx in range(len(headlines)):
    doc_tfidf = tfidf_matrix[idx].toarray()[0]
    
    # Calculate average TF-IDF score for the document
    # This gives us a measure of how "important" the document is based on TF-IDF
    avg_tfidf = np.mean(doc_tfidf[doc_tfidf > 0]) if np.any(doc_tfidf > 0) else 0.0
    
    # Get the SBERT embedding for this headline (from sbert_features computed earlier)
    sbert_embedding = sbert_features[idx]
    
    # Weight the SBERT embedding by the average TF-IDF score
    # Normalize to prevent extreme scaling
    weight = 1.0 + avg_tfidf  # Add 1 to ensure positive weight
    weighted_embedding = sbert_embedding * weight
    
    mixed_embeddings.append(weighted_embedding)

# Convert the result to a NumPy array
mixed_embeddings = np.array(mixed_embeddings)

# Normalize the embeddings
from sklearn.preprocessing import normalize
mixed_embeddings = normalize(mixed_embeddings, norm='l2')

print(f"Generated TF-IDF weighted SBERT feature matrix shape: {mixed_embeddings.shape}")

In [ ]:
def auto_find_eps(X, k_list=[3, 5, 7]):
    # search for best eps automatically
    all_eps_values = []

    plt.figure(figsize=(12, 4))

    for i, k in enumerate(k_list):
        nn = NearestNeighbors(n_neighbors=k)
        nn.fit(X)
        distances, indices = nn.kneighbors(X)
        k_distances = np.sort(distances[:, k - 1])

        kneedle = KneeLocator(range(len(k_distances)), k_distances,
                              curve='convex', direction='increasing')

        if kneedle.knee is not None:
            knee_eps = k_distances[kneedle.knee]
            all_eps_values.append(knee_eps)

            plt.subplot(1, 3, i + 1)
            plt.plot(k_distances, 'b-', linewidth=1)
            plt.axhline(y=knee_eps, color='red', linestyle='--',
                        label=f'eps: {knee_eps:.3f}')
            plt.xlabel('sample pouint in ascending order based on distance')
            plt.ylabel(f'distance of {k}th nearest neighbor ')
            plt.title(f'k={k}')
            plt.legend()
            plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    final_eps = np.median(all_eps_values)
    print(f"recommended eps: {final_eps:.3f}")
    return final_eps


def generate_stopwords_from_titles(all_headlines, doc_freq_threshold=0.05):
    doc_frequency = defaultdict(int)
    total_docs = len(all_headlines)

    for headline in all_headlines:
        words = set(re.findall(r'\b[a-zA-Z]{2,}\b', headline.lower()))
        for word in words:
            doc_frequency[word] += 1

    # select those words exceed doc_freq_threshold
    auto_stopwords = set()
    for word, doc_count in doc_frequency.items():
        doc_freq_ratio = doc_count / total_docs
        if doc_freq_ratio > doc_freq_threshold:
            auto_stopwords.add(word)
    
    # Always include common words that should not be topic keywords
    common_stopwords = {"the", "with", "and", "over", "after", "from", "into", "about", "more", "back", "out",
                        "new", "says", "not", "down", "off", "shows"}
    auto_stopwords.update(common_stopwords)

    print(f"generate{len(auto_stopwords)} stopping words")
    print(f"top 1 to 20 stopping words : {sorted(list(auto_stopwords))[:20]}")

    return auto_stopwords


def analyze_cluster_topics(df_results, all_headlines, top_words=5):
    # generate stopwords based on original dataset
    stopwords = generate_stopwords_from_titles(all_headlines)

    print(f"\n--- theme analysis ---")

    for label in sorted(df_results['cluster_label'].unique()):
        cluster_data = df_results[df_results['cluster_label'] == label]

        if label == -1:
            print(f"\n noise({len(cluster_data)}pages)")
            continue

        print(f"\n cluster {label} ({len(cluster_data)}pages):")

        # extract every words in the cluster
        cluster_words = []
        for headline in cluster_data['headline']:
            words = re.findall(r'\b[a-zA-Z]{3,}\b', headline.lower())
            filtered_words = [w for w in words if w not in stopwords]
            cluster_words.extend(filtered_words)

        # count word frequency and output theme
        if cluster_words:
            word_freq = Counter(cluster_words)
            top_keywords = word_freq.most_common(top_words)

            print("   theme key words", end=" ")
            keyword_list = []
            for word, count in top_keywords:
                keyword_list.append(f"{word}({count})")
            print(', '.join(keyword_list))


def run_complete_clustering(matrix, dates, headlines, matrix_name="LSA_tfidf"):
    print(f"---newstitles clustering analysis (using{matrix_name}matrix, Euclidean distance)---")

    # 1.search eps automatically
    eps_value = auto_find_eps(matrix)

    # 2.DBSCAN
    dbscan = DBSCAN(eps=0.2,
                    min_samples=5)
    cluster_labels = dbscan.fit_predict(matrix)

    # 3.creat DataFrame for results
    df_results = pd.DataFrame({
        'headline': headlines,
        'publish_date': pd.to_datetime(dates),
        'cluster_label': cluster_labels
    })

    # 4.show the result
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)

    print(f"\n clustering results :")
    print(f"   number of clusters: {n_clusters}")
    print(f"   noise: {n_noise}")
    print(f"   parameters: eps={eps_value:.3f}")

    # 5.using original data to do theme analysis
    analyze_cluster_topics(df_results, headlines)

    return df_results

def auto_find_eps_cosine(X, k_list=[5, 10, 15]):
    all_eps_values = []
    plt.figure(figsize=(12, 4))

    nn = NearestNeighbors(metric='cosine')

    for i, k in enumerate(k_list):
        nn.set_params(n_neighbors=k + 1)
        nn.fit(X)

        distances, indices = nn.kneighbors(X)
        k_distances = np.sort(distances[:, k])

        kneedle = KneeLocator(
            range(len(k_distances)),
            k_distances,
            curve='convex',
            direction='increasing'
        )

        if kneedle.knee is not None:
            eps = k_distances[kneedle.knee]
            all_eps_values.append(eps)

            plt.subplot(1, 3, i + 1)
            plt.plot(k_distances)
            plt.axhline(eps, color='r', linestyle='--')
            plt.title(f"k = {k}")
            plt.grid()

    plt.tight_layout()
    plt.show()

    final_eps = np.median(all_eps_values) if all_eps_values else 0.5
    print(f"Recommended eps (cosine): {final_eps:.3f}")
    return final_eps
    
def run_complete_clustering_cosine(X, dates, headlines, matrix_name="SBERT"):
    print(f"--- Clustering using {matrix_name} + cosine distance ---")

    eps_value = auto_find_eps_cosine(X)


    dbscan = DBSCAN(
        eps=0.5,
        min_samples=5,
        metric='cosine',
        n_jobs=-1
    )

    cluster_labels = dbscan.fit_predict(X)

    df_results = pd.DataFrame({
        'headline': headlines,
        'publish_date': pd.to_datetime(dates),
        'cluster_label': cluster_labels
    })

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)

    print("\nClustering results:")
    print(f"  number of clusters: {n_clusters}")
    print(f"  noise: {n_noise}")
    print(f"  parameters: eps={eps_value:.3f}")

    analyze_cluster_topics(df_results, headlines)
    return df_results
    

run_complete_clustering(tfidf_lsa_features, dates, headlines)
run_complete_clustering_cosine(sbert_features, dates, headlines, "SBERT")
run_complete_clustering_cosine(mixed_embeddings, dates, headlines, "Mixed")

##  SBERT → UMAP → HDBSCAN




In [ ]:
def reassign_noise_by_knn(embeddings, labels, k=10):
    
    from sklearn.neighbors import NearestNeighbors
    import numpy as np
    
    labels = labels.copy()
    noise_indices = np.where(labels == -1)[0]
    
    if len(noise_indices) == 0:
        return labels
    
    knn = NearestNeighbors(n_neighbors=k)
    knn.fit(embeddings)
    
    for idx in noise_indices:
        dist, neigh = knn.kneighbors([embeddings[idx]], n_neighbors=k)
        neigh_labels = labels[neigh[0]]
        
        valid = neigh_labels[neigh_labels != -1]
        if len(valid) > 0:
            new_label = Counter(valid).most_common(1)[0][0]
            labels[idx] = new_label
            
    return labels


def run_sbert_umap_hdbscan(sbert_features, dates, headlines):

    
    
    print("\n[Step 1] using UMAP to do dimension deduction...")
    print(f"  input dimension: {sbert_features.shape}")
    umap_reducer = umap.UMAP(
        n_components=15,          
        n_neighbors=15,          
        min_dist=0.0,             
        metric='cosine',          
        random_state=42,
        verbose=True
    )
    
    umap_embeddings = umap_reducer.fit_transform(sbert_features)
    print(f"  UMAP dimension deduction finished")
    print(f"  output demension {umap_embeddings.shape}")
    
    
    print("\n[Step 2] using HDBSCAN to do clustering...")
    
    
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size= 100,           
        min_samples= 2,                 
        cluster_selection_epsilon=0.12, 
        metric='euclidean',            
        cluster_selection_method='eom' 
    )
    
    cluster_labels = clusterer.fit_predict(umap_embeddings)
    print(f"  HDBSCAN clustering finished")
    
    initial_noise = list(cluster_labels).count(-1)
    if initial_noise > 0:
        print(f"\n[Step 2.5]  using k-NN to reallocate noise...")
        print(f"  original noise: {initial_noise} ({initial_noise/len(headlines)*100:.2f}%)")
        cluster_labels = reassign_noise_by_knn(umap_embeddings, cluster_labels, k=10)
        final_noise = list(cluster_labels).count(-1)
        reassigned = initial_noise - final_noise
        print(f"  reallocating finished")
        print(f"  reallocating noise: {reassigned} ({reassigned/initial_noise*100:.1f}% is reallocated)")
        print(f"  rest noise: {final_noise} ({final_noise/len(headlines)*100:.2f}%)")
    df_results = pd.DataFrame({
        'headline': headlines,
        'publish_date': pd.to_datetime(dates),
        'cluster_label': cluster_labels
    })
    
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)
    cluster_sizes = Counter(cluster_labels)
    
    print(f"\n[clustering result]")
    print(f"  total sample: {len(headlines):,}")
    print(f"  clusters: {n_clusters}")
    print(f"  noise: {n_noise} ({n_noise/len(headlines)*100:.2f}%)")
    print(f"  cluster samples: {len(headlines)-n_noise:,} ({(len(headlines)-n_noise)/len(headlines)*100:.2f}%)")
    
    
    print(f"\n[distribution of clusters]")
    sorted_clusters = sorted([(label, size) for label, size in cluster_sizes.items() if label != -1], 
                            key=lambda x: x[1], reverse=True)
    print(f"  max cluster: {sorted_clusters[0][1]} samples (cluster {sorted_clusters[0][0]})")
    print(f"  min cluster: {sorted_clusters[-1][1]} samples (cluster {sorted_clusters[-1][0]})")
    print(f"  average size: {np.mean([size for _, size in sorted_clusters]):.1f} samples")
    
    
    print(f"\n[Step 4.5] calculating Silhouette Coefficient...")
    
    from sklearn.metrics import silhouette_score, silhouette_samples
    
    clustered_mask = cluster_labels != -1
    if clustered_mask.sum() > 1 and n_clusters > 1:
       
        clustered_embeddings = umap_embeddings[clustered_mask]
        clustered_labels = cluster_labels[clustered_mask]
        silhouette_avg = silhouette_score(clustered_embeddings, clustered_labels, metric='euclidean')
        print(f"  Silhouette Coefficient: {silhouette_avg:.4f}")
        
        
        sample_silhouette_values = silhouette_samples(clustered_embeddings, clustered_labels, metric='euclidean')
        cluster_silhouette_scores = {}
        for label in sorted(set(clustered_labels)):
            cluster_mask = clustered_labels == label
            cluster_silhouette_scores[label] = sample_silhouette_values[cluster_mask].mean()
            print(f"    Cluster {label}: {cluster_silhouette_scores[label]:.4f} (samples: {cluster_mask.sum()})")
        


    print(f"\n" + "="*60)
    analyze_cluster_topics(df_results, headlines)

    return df_results, umap_embeddings, clusterer, silhouette_avg


df_results_best, umap_embeddings, clusterer, silhouette_avg = run_sbert_umap_hdbscan(sbert_features, dates, headlines)


In [ ]:

print("\n" + "="*60)
print("clustering visualization...")
print("="*60)
print("\n[1] generating UMAP 2D projection...")
umap_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)
umap_2d_embeddings = umap_2d.fit_transform(sbert_features)
print("  UMAP dimension deduction finished")
fig, ax1 = plt.subplots(figsize=(12, 10))
noise_mask = df_results_best['cluster_label'] == -1
cluster_mask = ~noise_mask
if cluster_mask.sum() > 0:
    scatter1 = ax1.scatter(umap_2d_embeddings[cluster_mask, 0],
                          umap_2d_embeddings[cluster_mask, 1],
                          c=df_results_best.loc[cluster_mask, 'cluster_label'],
                          cmap='tab20', s=2, alpha=0.6)
if noise_mask.sum() > 0:
    ax1.scatter(umap_2d_embeddings[noise_mask, 0],
               umap_2d_embeddings[noise_mask, 1],
               c='lightgray', s=0.5, alpha=0.3, label='Noise')
ax1.set_title('UMAP 2D Projection - Clusters', fontsize=12, fontweight='bold')
ax1.set_xlabel('UMAP Dimension 1')
ax1.set_ylabel('UMAP Dimension 2')
if cluster_mask.sum() > 0:
    plt.colorbar(scatter1, ax=ax1, label='Cluster Label')
ax1.legend()
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
if silhouette_avg is not None:
    print("\n" + "="*60)
    print("generating Silhouette Coefficient visualization ..")
    print("="*60)
    from sklearn.metrics import silhouette_samples
    clustered_mask = df_results_best['cluster_label'] != -1
    if clustered_mask.sum() > 1:
        clustered_embeddings = umap_embeddings[clustered_mask]
        clustered_labels = df_results_best.loc[clustered_mask, 'cluster_label'].values
        sample_silhouette_values = silhouette_samples(clustered_embeddings, clustered_labels, metric='euclidean')
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        y_lower = 10
        unique_labels = sorted(set(clustered_labels))
        colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
        
        for i, label in enumerate(unique_labels):
            cluster_mask = clustered_labels == label
            cluster_silhouette_values = sample_silhouette_values[cluster_mask]
            cluster_silhouette_values.sort()
            
            size_cluster_i = cluster_silhouette_values.shape[0]
            y_upper = y_lower + size_cluster_i
            
            color = colors[i]
            ax1.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_values,
                             facecolor=color, edgecolor=color, alpha=0.7)
            ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, f'Cluster {label}')
            y_lower = y_upper + 10
        
        ax1.axvline(x=silhouette_avg, color="red", linestyle="--", linewidth=2, label=f'average Silhouette Coefficient: {silhouette_avg:.4f}')
        ax1.set_xlabel('Silhouette Coefficient')
        ax1.set_ylabel('cluster labels')
        ax1.set_title('distribution of Silhouette Coefficient of every cluster', fontsize=12, fontweight='bold')
        ax1.set_ylim([0, len(clustered_labels) + (len(unique_labels) + 1) * 10])
        ax1.set_xlim([-0.1, 1])
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        ax2.hist(sample_silhouette_values, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
        ax2.axvline(x=silhouette_avg, color="red", linestyle="--", linewidth=2, 
                   label=f'average Silhouette Coefficient: {silhouette_avg:.4f}')
        ax2.set_xlabel('Silhouette Coefficient')
        ax2.set_ylabel('samples')
        ax2.set_title('Silhouette Coefficient Distribution Histogram', fontsize=12, fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n Silhouette Coefficient visualization finished！")
        print(f"  overall Silhouette Coefficient: {silhouette_avg:.4f}")

print("\n viusalization finished！")
